In [1]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "../dtgraph"))

In [2]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [3]:
from type_checking.environment import Environment

env = Environment("../../dtgraph/type_checking/ENVs/env_fraud_experimentation.json")

In [4]:
from pg_schema.loader import SchemaLoader

source_schema_path = "../../dtgraph/pg_schema/schemas/schema_fraud_source.json"
target_schema_path = "../../dtgraph/pg_schema/schemas/schema_fraud_target.json"

source_schema = SchemaLoader(
    source_schema_path,
    env=env,
    section="source"
)

target_schema = SchemaLoader(
    target_schema_path,
    env=env,
    section="target"
)

##### Rules

In [5]:
Rule1 = Rule(
    """
MATCH (c:Client)
WHERE NOT c:Mule
OPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)
OPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)
OPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)
WITH 
    c,
    collect(DISTINCT e.email) AS emails,
    collect(DISTINCT p.phoneNumber) AS phones,
    collect(DISTINCT s.ssn) AS ssns
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name,
    name_camel_case = apoc.text.upperCamelCase(c.name),
    email = head(emails),
    phone = head(phones),
    ssn = head(ssns)
})
""",
    env=env,
    type_strict=True,
)

Rule2 = Rule(
    """
MATCH (c:Client:Mule)
OPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)
OPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)
OPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)
WITH 
    c,
    collect(DISTINCT e.email) AS emails,
    collect(DISTINCT p.phoneNumber) AS phones,
    collect(DISTINCT s.ssn) AS ssns
GENERATE
(p = (c.id):Person,Scammer {
    id = c.id,
    name = c.name,
    name_camel_case = apoc.text.upperCamelCase(c.name),
    email = head(emails),
    phone = head(phones),
    ssn = head(ssns)
})
""",
    env=env,
    type_strict=True,
)

Rule3 = Rule(
"""
MATCH (c:Client)-[:PERFORMED]->(t:CashIn)
WITH c, collect(t.amount) AS amounts
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name
})
-[():AGGREGATED_CASHIN]->(
    agg = (c.id, "cashin"):CashInSummary {
        all_amounts = amounts,
        count = size(amounts)
    }
)
""",
env=env,
type_strict=True,
)

Rule4 = Rule(
"""
MATCH (c:Client)-[:PERFORMED]->(t:CashOut)
WITH c, collect(t.amount) AS amounts
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name
})
-[():AGGREGATED_CASHOUT]->(
    agg = (c.id, "cashout"):CashOutSummary {
        all_amounts = amounts,
        count = size(amounts)
    }
)
""",
env=env,
type_strict=True,
)

Rule5 = Rule(
"""
MATCH (c:Client)-[:PERFORMED]->(t:Transfer)
WITH c, collect(t.amount) AS amounts
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name
})
-[():AGGREGATED_TRANSFER]->(
    agg = (c.id, "transfer"):TransferSummary {
        all_amounts = amounts,
        count = size(amounts)
    }
)
""",
env=env,
type_strict=True,
)

Rule6 = Rule(
"""
MATCH (c:Client)-[:PERFORMED]->(t:Payment)
WITH c, collect(t.amount) AS amounts
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name
})
-[():AGGREGATED_PAYMENT]->(
    agg = (c.id, "payment"):PaymentSummary {
        all_amounts = amounts,
        count = size(amounts)
    }
)
""",
env=env,
type_strict=True,
)

Rule7 = Rule(
"""
MATCH (c:Client)-[:PERFORMED]->(t:Debit)
WITH c, collect(t.amount) AS amounts
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name
})
-[():AGGREGATED_DEBIT]->(
    agg = (c.id, "debit"):DebitSummary {
        all_amounts = amounts,
        count = size(amounts)
    }
)
""",
env=env,
type_strict=True,
)

Rule8 = Rule(
"""
MATCH (c:Client)-[:PERFORMED]->(t1:Transaction)-[:NEXT]->(t2:Transaction)
OPTIONAL MATCH (t2)-[:TO]->(target)
WITH c, t1, t2, target
GENERATE
(f = (c.id, t1.id, t2.id):TransactionFlow {
    from_amount = t1.amount,
    to_amount = t2.amount,
    increasing = t2.amount > t1.amount
})
-[():STARTED_BY]-> (p = (c.id):),
(f)-[():FROM_TX]-> (tx1 = (t1.id):NewTransaction {
    id = t1.id,
    amount = t1.amount
}),
(f)-[():TO_TX]-> (tx2 = (t2.id):NewTransaction {
    id = t2.id,
    amount = t2.amount
}),
(f)-[():TARGET]-> (dest = (target.id):Destination)
""",
env=env,
type_strict=True,
)


--- Checking Rule ---
{'lhs': 'MATCH (c:Client)\nWHERE NOT c:Mule\nOPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)\nOPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)\nOPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)\nWITH \n    c,\n    collect(DISTINCT e.email) AS emails,\n    collect(DISTINCT p.phoneNumber) AS phones,\n    collect(DISTINCT s.ssn) AS ssns', 'constructors': [{'alias': 'p', 'ids': ['c.id'], 'labels': ['Person'], 'properties': [{'key': 'id', 'value': 'c.id'}, {'key': 'name', 'value': 'c.name'}, {'key': 'name_camel_case', 'value': 'apoc.text.upperCamelCase(c.name)'}, {'key': 'email', 'value': 'head(emails)'}, {'key': 'phone', 'value': 'head(phones)'}, {'key': 'ssn', 'value': 'head(ssns)\n'}]}]}

 Type checking passed


--- Checking Rule ---
{'lhs': 'MATCH (c:Client:Mule)\nOPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)\nOPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)\nOPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)\nWITH \n    c,\n    collect(DISTINCT e.email) AS emails,\n    collect(DISTINCT p.phoneNumber) AS p

##### Schema Conformance

In [7]:
from dtgraph.pg_schema.check_schema import check_schema

check_schema(
    [
        Rule1,
        Rule2,
        Rule3,
        Rule4,
        Rule5,
        Rule6,
        Rule7,
        Rule8
    ],
    target_schema,
)


--- Checking Rule ---
Compiled:
MATCH (c:Client)
WHERE NOT c:Mule
OPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)
OPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)
OPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)
WITH 
    c,
    collect(DISTINCT e.email) AS emails,
    collect(DISTINCT p.phoneNumber) AS phones,
    collect(DISTINCT s.ssn) AS ssns
MERGE (p:_dummy {
    _id: "(" + c.id + ")" 
})
ON CREATE
    SET p:Person,
        p.id = c.id,
        p.name = c.name,
        p.name_camel_case = apoc.text.upperCamelCase(c.name),
        p.email = head(emails),
        p.phone = head(phones),
        p.ssn = head(ssns)
ON MATCH
    SET p:Person,
        p.`id.conflict` = 
        CASE
            WHEN p.id = "Conflict Detected!" THEN
                coalesce(p.`id.conflict`, []) + CASE
            WHEN c.id IS NULL THEN []
            WHEN valueType(c.id) STARTS WITH 'LIST' THEN c.id
            ELSE [c.id]
        END
            WHEN p.id IS NOT NULL AND p.id <> c.id THEN
                coalesce(p.`id.con

##### Applying Rules

In [6]:
my_transform = Transformation(
    [
        Rule1,
        Rule2,
        Rule3,
        Rule4,
        Rule5,
        Rule6,
        Rule7,
        Rule8
    ]
)
my_transform.apply_on(graph)

Index: Added 1 index, completed after 12 ms.
Rule: Added 4000 labels, created 2000 nodes, set 14000 properties, created 0 relationships, completed after 1058 ms.
Rule: Added 1299 labels, created 433 nodes, set 3031 properties, created 0 relationships, completed after 884 ms.
Rule: Added 3958 labels, created 1979 nodes, set 11874 properties, created 1979 relationships, completed after 720 ms.
Rule: Added 3884 labels, created 1942 nodes, set 11652 properties, created 1942 relationships, completed after 654 ms.
Rule: Added 4360 labels, created 2180 nodes, set 13080 properties, created 2180 relationships, completed after 574 ms.
Rule: Added 3892 labels, created 1946 nodes, set 11676 properties, created 1946 relationships, completed after 612 ms.
Rule: Added 2678 labels, created 1339 nodes, set 8034 properties, created 1339 relationships, completed after 450 ms.
Rule: Added 1291592 labels, created 644663 nodes, set 4177390 properties, created 1284628 relationships, completed after 29458 ms.

34410

##### Abort Transformation

In [8]:
my_transform.abort()

Index: Removed 1 index, completed after 44 ms.


Abort: Deleted 656482 nodes, deleted 1294014 relationships, completed after 5880 ms.
